# Piano B — Notebook J: figure della tesi

Genera tutte le figure (in inglese, PDF e PNG) dai risultati di **tutte** le sessioni, più `INDEX.md` con
una descrizione e i numeri principali di ogni figura. Nessun training: basta la CPU (circa 10-20 minuti).
Le figure che dipendono da run non ancora fatte (notebook K, voti dei notebook H/I) si saltano da sole e
compaiono quando i loro output vengono collegati.

Impostazioni: Accelerator **None (CPU)**, Internet off. Input: `pianob-code` e **l'ultimo output di ogni
notebook** (A, B, C, D, E, F, G, e quando ci sono H, I, K). Gli output sono cumulativi: basta l'ultimo di
ciascun notebook; collegarne di più vecchi non fa danni (i duplicati si scartano). Output:
`figures/` e `figures_thesis.zip`.

In [ ]:
import os
os.environ["PIANOB_NO_RESTORE"] = "1"      # gli output collegati si leggono, non si copiano nello stato

In [ ]:
# Setup: trova il codice (Dataset "pianob-code"), lo copia fuori da /kaggle/working (così non finisce
# negli output) e lo aggiunge a sys.path. Non serve modificare nulla qui.
import glob, os, shutil, sys, time
os.environ.setdefault("PIANOB_T0", str(time.time()))      # inizio della sessione (guardia delle 12 h)
os.environ.setdefault("PIANOB_BUDGET_H", "11.3")           # stop pulito prima del limite Kaggle
SYNTHETIC = os.environ.get("PIANOB_SYNTHETIC") == "1"      # solo per prove locali


def _find_code():
    cands = []
    roots = ["/kaggle/input"] if os.path.exists("/kaggle/input") else []
    roots += [os.getcwd(), os.path.dirname(os.getcwd())]
    for root in roots:
        for p in glob.glob(os.path.join(root, "**", "pianob", "__init__.py"), recursive=True):
            base = os.path.dirname(os.path.dirname(p))
            if "pianob_state" not in base:
                cands.append(base)
    assert cands, "Codice non trovato: aggiungi il Dataset 'pianob-code' come input (Add Input)."
    # se ci sono più copie, usa quella con VERSION più recente
    def _ver(b):
        v = os.path.join(b, "VERSION")
        return open(v).read().strip() if os.path.exists(v) else ""
    return sorted(cands, key=_ver)[-1]


CODE_SRC = _find_code()
if os.path.exists("/kaggle/working"):
    CODE = "/tmp/pianob_code"
    shutil.rmtree(CODE, ignore_errors=True)
    shutil.copytree(CODE_SRC, CODE, ignore=shutil.ignore_patterns("notebooks", "*.zip", "local_work"))
else:
    CODE = CODE_SRC
sys.path.insert(0, CODE)
os.chdir(CODE)
print("codice da", CODE_SRC, "| versione", open(os.path.join(CODE, "VERSION")).read().strip()
      if os.path.exists(os.path.join(CODE, "VERSION")) else "(locale)")

from pianob import kaggle_env, nb_helpers  # noqa: E402
nb_helpers.print_env()
kaggle_env.restore_state_from_inputs()

In [ ]:
from pathlib import Path
sys.path.insert(0, os.path.join(CODE, "tools"))
from aggregate import load_inputs
from pianob import figures

states = sorted({p for r in kaggle_env.input_roots() for p in r.rglob("pianob_state") if p.is_dir()})
zips = sorted({z for r in kaggle_env.input_roots() for z in r.rglob("*results_*.zip")
               if not any(s.parent == z.parent for s in states)})
INPUTS = [str(p) for p in states + zips]
print(f"{len(INPUTS)} input con risultati:")
for p in INPUTS:
    print("  ", p)
assert INPUTS or SYNTHETIC, "Nessun output collegato: aggiungi come input l'ultimo output di ogni notebook (A-G)."

In [ ]:
OUT = kaggle_env.working_dir() / "figures"
if INPUTS:
    df, jobs, _ = load_inputs(INPUTS)
    made = figures.make_all(df, jobs, INPUTS, OUT)
    print((OUT / "INDEX.md").read_text())
    shutil.make_archive(str(kaggle_env.working_dir() / "figures_thesis"), "zip", OUT)
    print(f"\n{len(made)} figure -> figures_thesis.zip")